# 3. 연결부만 양자화한 모델에서 공격 전이 확인

`1_공격사진_생성_및_원본모델_확인.ipynb`에서 만든 공격 사진(`adv/*.png`)을 **그대로** 읽어,
연결부(비전 인코더의 출력을 언어모델로 넘기는 Linear 층 1개)만 양자화했을 때 답 길이가 어떻게 되는지 잰다.
이 노트북은 공격 사진을 만들거나 바꾸지 않는다.

- 넣은 경우: 기준(bf16, 양자화 없음), 연결부만 8비트, 연결부만 4비트
- 평가 방법과 함수는 `2_비전인코더_양자화_전이확인.ipynb`(비전 인코더)와 같고, `CASES`와 결과 파일 이름만 다르다.

**실행 방법**
- 로컬(VSCode): 커널로 `.venv`를 고르고 전부 실행
- 터미널: `jupyter execute "3_연결부_양자화_전이확인.ipynb" --inplace --timeout=-1` (결과는 이 파일의 각 셀 아래에 저장된다)

**결과가 남는 곳**
- 각 셀 아래의 표
- `results/3_연결부_양자화_전이확인.csv` (실행할 때마다 새로 쓴다)

In [ ]:
# 코랩에서만 패키지를 설치한다. 로컬은 README의 방법으로 만든 .venv를 쓴다.
import sys
if "google.colab" in sys.modules:
    !pip install -q -U transformers accelerate bitsandbytes
    !pip install -q --force-reinstall --no-cache-dir "pillow==11.2.1"

import torch, transformers, bitsandbytes, PIL
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| bitsandbytes", bitsandbytes.__version__, "| pillow", PIL.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")

## 설정

- `PARTS`: 양자화 대상이 되는 Linear 층이 들어 있는 네 부분과, 그 부분의 모듈 이름
- `CASES`: 실행할 경우. `(부분, 비트)`이고 `(None, None)`은 양자화 없는 기준이다.

bitsandbytes는 "양자화할 부분"이 아니라 "건너뛸 부분"을 받는다. 그래서 한 부분만 양자화하려면 나머지 세 부분을 모두 건너뛰게 한다.
4비트는 nf4 방식, 계산은 bf16으로 고정한다. 양자화하지 않는 부분은 bf16 그대로다.

In [ ]:
MID = "HuggingFaceTB/SmolVLM-256M-Instruct"
Q = "Describe this image in one sentence."
MAX_NEW = 512

PARTS = {
    "V": ("비전 인코더", "model.vision_model"),
    "C": ("연결부",     "model.connector"),
    "L": ("언어모델",   "model.text_model"),
    "H": ("출력층",     "lm_head"),
}

CASES = [
    (None, None),   # 기준: 양자화 없음(bf16)
    ("C", 8),       # 연결부만 8비트
    ("C", 4),       # 연결부만 4비트
]

def case_name(part, bits):
    return "기준 bf16" if part is None else f"{PARTS[part][0]}만 {bits}비트"

In [ ]:
import os, gc, csv, logging
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig

dev = "cuda"
proc = AutoProcessor.from_pretrained(MID); proc.image_processor.do_image_splitting = False
QUANT_TYPES = ("Linear4bit", "Linear8bitLt")

def load_case(part, bits):
    if part is None:
        return AutoModelForImageTextToText.from_pretrained(MID, dtype=torch.bfloat16).to(dev).eval()
    skip = [key for p, (_, key) in PARTS.items() if p != part]   # 대상이 아닌 부분은 전부 건너뛴다
    if bits == 8:
        cfg = BitsAndBytesConfig(load_in_8bit=True, llm_int8_skip_modules=skip)
    elif bits == 4:
        cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                 bnb_4bit_compute_dtype=torch.bfloat16, llm_int8_skip_modules=skip)
    else:
        raise ValueError("bitsandbytes는 8비트와 4비트만 지원한다")
    return AutoModelForImageTextToText.from_pretrained(
        MID, quantization_config=cfg, dtype=torch.bfloat16, device_map=dev).eval()

def census(model):
    """부분별 [양자화된 Linear 수, 전체 Linear 수]"""
    c = {p: [0, 0] for p in PARTS}
    for name, m in model.named_modules():
        t = type(m).__name__
        if t != "Linear" and t not in QUANT_TYPES:
            continue
        owner = [p for p, (_, key) in PARTS.items() if name == key or name.startswith(key + ".")]
        assert len(owner) == 1, f"어느 부분인지 알 수 없는 Linear: {name}"
        c[owner[0]][1] += 1
        c[owner[0]][0] += t in QUANT_TYPES
    return c

def check_census(c, part):
    """의도한 부분만 전부 양자화됐는지 확인한다. 아니면 멈춘다."""
    for p, (q, total) in c.items():
        want = total if p == part else 0
        assert q == want, f"{PARTS[p][0]}: 양자화된 Linear {q}개 (기대 {want}개) → 건너뛸 이름을 확인할 것"

def load512(path):
    return Image.open(path).convert("RGB").resize((512, 512), Image.BICUBIC)

@torch.no_grad()
def gen_len(model, path):
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": Q}]}]
    t = proc.apply_chat_template(msgs, add_generation_prompt=True)
    enc = proc(text=t, images=[load512(path)], return_tensors="pt").to(dev)
    out = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False)
    return out.shape[1] - enc["input_ids"].shape[1]

## 실행

경우마다 모델을 올리고, 부분별로 양자화된 Linear 수를 출력한 뒤, 원본 사진과 공격 사진의 답 길이를 잰다.
의도한 부분만 양자화되지 않았으면 그 자리에서 멈춘다.

In [ ]:
files = sorted(f for f in os.listdir("imgs") if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")))
stems = [os.path.splitext(f)[0] for f in files]
missing = [s for s in stems if not os.path.exists(f"adv/{s}.png")]
assert not missing, f"공격 사진이 없다: {missing} → 1번 노트북을 먼저 실행할 것"

res = {}   # (경우, 사진) → (원본 사진 답 길이, 공격 사진 답 길이)
for part, bits in CASES:
    name = case_name(part, bits)
    model = load_case(part, bits)
    c = census(model)
    check_census(c, part)
    print(f"=== {name} | 양자화된 Linear: " + ", ".join(f"{PARTS[p][0]} {q}/{t}" for p, (q, t) in c.items()))
    for f, s in zip(files, stems):
        res[(name, s)] = (gen_len(model, f"imgs/{f}"), gen_len(model, f"adv/{s}.png"))
        print(f"    {s:6s} 원본 {res[(name, s)][0]:4d} → 공격 {res[(name, s)][1]:4d}")
    del model; gc.collect(); torch.cuda.empty_cache()

## 요약

- 칸의 값은 `원본 사진 답 길이 → 공격 사진 답 길이`이다. 512는 `max_new_tokens` 상한에 닿았다는 뜻이다.
- "공격 유지"는 공격 사진이 상한에 닿은 사진 수, "원본도 상한"은 공격 없이도 상한에 닿은 사진 수다.

In [ ]:
names = [case_name(p, b) for p, b in CASES]
print("사진   | " + " | ".join(f"{n:>16s}" for n in names))
for s in stems:
    print(f"{s:6s} | " + " | ".join(f"{res[(n, s)][0]:>7d} → {res[(n, s)][1]:<6d}" for n in names))

print()
for n in names:
    keep = sum(res[(n, s)][1] >= MAX_NEW for s in stems)
    nat = sum(res[(n, s)][0] >= MAX_NEW for s in stems)
    print(f"{n}: 공격 유지 {keep}/{len(stems)}, 원본도 상한 {nat}/{len(stems)}")

os.makedirs("results", exist_ok=True)
out_csv = "results/3_연결부_양자화_전이확인.csv"
with open(out_csv, "w", newline="", encoding="utf-8-sig") as fp:
    w = csv.writer(fp)
    w.writerow(["case", "part", "bits", "image", "clean_len", "adv_len"])
    for part, bits in CASES:
        for s in stems:
            o, a = res[(case_name(part, bits), s)]
            w.writerow([case_name(part, bits), part or "none", bits or "bf16", s, o, a])
print("\n저장:", out_csv)

## 답변 내용 확인 (일부만)

위 표는 답 길이만 보여 준다. 여기서는 전체 사진 중 일부를 무작위로 뽑아, 모델이 실제로 뭐라고 답했는지 본다.

- 전체 사진이 n장이면 그중 `SHOW_N`장(기본 5장)을 무작위로 뽑는다. n이 5보다 작으면 전부 본다.
- 뽑힌 사진은 실행할 때마다 달라진다. 같은 사진을 다시 보려면 `SHOW_SEED`에 숫자를 넣는다.
- 볼 경우는 `SHOW_CASES`에서 고른다. 뽑힌 사진은 모든 경우에서 같으므로 경우끼리 비교할 수 있다.
- 위 실행 셀은 길이만 저장하므로, 이 셀은 고른 경우의 모델을 다시 올려 답을 다시 생성한다. 조건이 같아서 길이는 위 표와 같아야 하고, 다르면 `※` 표시가 붙는다.
- 위의 설정·함수·실행 셀을 먼저 실행한 뒤에 돌린다.

In [ ]:
import random

# ── 볼 대상 고르기 ──────────────────────────────────────────────
SHOW_N = 5                # 전체 사진 중 무작위로 몇 장을 볼지 (사진이 그보다 적으면 전부)
SHOW_SEED = None          # None이면 실행할 때마다 다르게 뽑는다. 숫자를 넣으면 매번 같은 사진을 뽑는다
SHOW_CASES = CASES        # 답을 볼 경우. 예: [(None, None), ("V", 4)]
SHOW_CHARS = 300          # 답을 앞에서 몇 글자까지 보여줄지 (None이면 전부)

SHOW_IMAGES = sorted(random.Random(SHOW_SEED).sample(stems, min(SHOW_N, len(stems))))
print(f"전체 {len(stems)}장 중 {len(SHOW_IMAGES)}장을 무작위로 뽑음: {SHOW_IMAGES}")

@torch.no_grad()
def gen_text(model, path):
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": Q}]}]
    t = proc.apply_chat_template(msgs, add_generation_prompt=True)
    enc = proc(text=t, images=[load512(path)], return_tensors="pt").to(dev)
    out = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False)
    new = out[0, enc["input_ids"].shape[1]:]
    return len(new), proc.tokenizer.decode(new, skip_special_tokens=True).strip()

def clip(text):
    return text if SHOW_CHARS is None or len(text) <= SHOW_CHARS else text[:SHOW_CHARS] + " …(이하 생략)"

for part, bits in SHOW_CASES:
    name = case_name(part, bits)
    model = load_case(part, bits)
    check_census(census(model), part)
    print(f"\n=== {name} ===")
    for s in SHOW_IMAGES:
        f = files[stems.index(s)]
        for i, (tag, path) in enumerate([("원본", f"imgs/{f}"), ("공격", f"adv/{s}.png")]):
            n, text = gen_text(model, path)
            before = res.get((name, s))
            note = "" if before is None or before[i] == n else f"  ※ 위 표의 값({before[i]})과 다름"
            print(f"[{s} | {tag} 사진 | {n}토큰]{note}\n    {clip(text)}")
    del model; gc.collect(); torch.cuda.empty_cache()